# Open Diagnostic Cyberdeck — Colab GPU Community Extraction (Strategy A)

Extracts Reddit + forum diagnostic threads for the **12 Strategy A makes** (`chevrolet`, `toyota`, `ford`, `honda`, `jeep`, `nissan`, `dodge`, `acura`, `hyundai`, `lexus`, `kia`, `gmc`) using **vLLM** with guided JSON decoding, strict quote verification, local-SSD SQLite (never SQLite over Google Drive FUSE!), and automatic Drive checkpointing.

### Compute Budget Guardrail: **160 Compute Units Max** (20 CUs YouTube + 20 CUs First-Aid Reserved)
* **Hard budget cap:** `CU_BUDGET = 160.0` CUs total across all sessions for Reddit/forums (tracked in `MyDrive/Cyberdeck_Community_DB/logs/cu_budget.json`).
* **Reserved:** **20 CUs** for YouTube repair-transcript extraction on Colab + **20 CUs** untouched for Pocket First-Aid Librarian mini-model training (`fasr` + `whisper_lora` + `vr` voice router).
* Cell 4 automatically computes remaining wall-clock minutes from your live `CU_PER_HOUR` rate, stops cleanly before hitting 160 CUs, and Cell 5 unassigns the runtime.

### Recommended Two-Tier Run (Best Accuracy-per-Credit inside 180 CUs)
1. **Calibration (Cell 3, < 1 CU):** Run 500 threads on `qwen3.5-27b` (or `gemma-4-31b`) vs `qwen3.5-9b` to verify live `threads/hr`, quote-fail %, and check the exact `CU/hr` shown in Colab's **Resources** panel (then plug that into `CU_PER_HOUR_OVERRIDE` in Cell 1).
2. **Tier 1 (`PRIORITY_ONLY = True`, ~62k high-signal outcome+code/test threads):** Use `qwen3.5-27b` (or `qwen2.5-32b` / `gemma-4-31b`) on A100-80GB / H100 (~60–95 CUs).
3. **Tier 2 (`PRIORITY_ONLY = False`, remaining ~146k threads):** Switch to `qwen3.5-9b` (BF16 on A100 or FP8 on L4/H100) to finish as many remaining threads as fit inside the remaining ~85–120 CUs (completed Tier 1 threads are skipped automatically!).
   * *Estimated total inside **180 CUs**:* **~135k–235k threads** with `qwen3.5-9b` (~72–105 hrs on L4 or ~15–32 hrs on A100), or **~62k Tier-1 heavyweight + ~70k–130k Tier-2 9B threads** (~130k–195k total).

### Folder layout on Google Drive (`MyDrive/Cyberdeck_Community_DB/`)
* `colab_worker.py` — extraction + quote validator (exact parity with `kb/build_community.py`)
* `inputs/<make>.work.db.gz` — prefiltered thread inputs (414 MB total from `stage_strategy_a.sh`)
* `checkpoints/<make>.extract.jsonl` — append-only resume checkpoints synced every 500 threads
* `work_dbs/<make>.work.db` — finished per-make SQLite DBs saved after each make completes
* `logs/cu_budget.json` — cumulative Compute Unit ledger across resumed sessions

In [ ]:
# === CELL 1: Mount Google Drive, detect GPU, enforce 180 CU budget cap, and pick model preset ===
import json, os, subprocess, time, torch
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Cyberdeck_Community_DB'
for sub in ('inputs', 'checkpoints', 'work_dbs', 'logs'):
    os.makedirs(f'{DRIVE_DIR}/{sub}', exist_ok=True)

# --- BUDGET GUARDRAIL (160 CUs Reddit/forums; 20 CUs YouTube + 20 CUs First-Aid reserved) ---
CU_BUDGET = 160.0            # Hard cap across all Reddit/forum sessions
CU_PER_HOUR_OVERRIDE = None  # After Cell 3, set to the exact CU/hr shown in Colab's Resources panel (e.g. 2.1 or 7.5)
STARTUP_OVERHEAD_CU = 0.5    # Conservative buffer for pip install + model download per session

# --- MODEL & TIER CONFIGURATION ---
# Recommended Two-Tier Strategy (fits cleanly inside 180 CUs):
#   Tier 1: MODEL_PRESET = 'qwen3.5-27b' (or 'gemma-4-31b' / 'qwen2.5-32b'), PRIORITY_ONLY = True  (~62k outcome+code/test threads)
#   Tier 2: MODEL_PRESET = 'qwen3.5-9b',                                     PRIORITY_ONLY = False (remaining ~146k threads)
#
# All supported presets:
#   'qwen3.5-27b'         -> Qwen/Qwen3.5-27B (dense/hybrid flagship; needs A100-80GB/H100, or L4 with FP8)
#   'qwen3.5-9b'          -> Qwen/Qwen3.5-9B (best speed/credit: ~135k-235k thr / 180 CUs; BF16 on A100, FP8 on L4/H100)
#   'qwen2.5-32b'         -> Qwen/Qwen2.5-32B-Instruct (battle-tested dense JSON extractor; A100-80GB/H100)
#   'gemma-4-31b'         -> google/gemma-4-31b-it (Apache 2.0 US dense 31B; A100-80GB/H100)
#   'gemma-4-26b-moe'     -> google/gemma-4-26b-a4b-it (Apache 2.0 US 26B MoE, ~3.8B active; fast on A100/L4)
#   'nemotron-3-nano-30b' -> nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16 (US hybrid MoE, ~3B active; A100-80GB/H100)
#   'gpt-oss-20b'         -> openai/gpt-oss-20b (Apache 2.0 US MXFP4 MoE, ~13 GB VRAM; reasoning-heavy, test in Cell 3 first!)
#   'inkling-2-mini'      -> thinkingmachines/Inkling-2-Mini (Nov 2026 US 26.4B/3.4B-active looped MoE; ~20 GB VRAM, test in Cell 3 first!)
#   'qwen3-next-80b-fp8'  -> Qwen/Qwen3-Next-80B-A3B-Instruct (needs H100/A100-80GB + quantized checkpoint; test in Cell 3 first!)
MODEL_PRESET = 'qwen3.5-27b'
PRIORITY_ONLY = False  # Tier 1 done across all 26 makes! Now backfilling general threads (~61 CUs left in 160 CU cap)

# Optional additional per-session cap in minutes (0 = governed automatically by remaining 180 CU budget)
MAX_SESSION_MINUTES = 0

# Auto-unassign Colab runtime when finished or when 180 CU cap is reached so idle GPUs never drain Compute Units
AUTO_SHUTDOWN = True

MAKES = [
    # Core everyday US/PNW mobile-mechanic makes (Subaru already 100% done!)
    'chevrolet', 'toyota', 'honda', 'ford', 'nissan', 'jeep', 'dodge', 'hyundai', 'kia', 'gmc',
    # Common secondary / indie-shop / PNW makes
    'mazda', 'volkswagen', 'acura', 'lexus', 'bmw', 'volvo', 'audi', 'mercedes_benz',
    # Common domestic/import extras (small DBs; older/discontinued makes omitted so budget cuts off gracefully)
    'chrysler', 'buick', 'cadillac', 'lincoln', 'scion', 'infiniti', 'mitsubishi', 'mini',
]

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9) if torch.cuda.is_available() else 0
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
has_native_fp8 = cap >= (8, 9)  # L4 (8.9), H100 (9.0); A100 is 8.0 (no native FP8)
assert torch.cuda.is_available(), 'No GPU attached! Switch Runtime -> Change runtime type -> L4 or A100.'

# Conservative default CU/hr estimates by GPU (override with live rate from Colab Resources panel after Cell 3!)
if CU_PER_HOUR_OVERRIDE is not None:
    EST_CU_PER_HOUR = float(CU_PER_HOUR_OVERRIDE)
elif 'H100' in gpu_name:
    EST_CU_PER_HOUR = 18.5
elif 'A100' in gpu_name:
    EST_CU_PER_HOUR = 12.0 if vram_gb >= 75 else 7.5
elif 'L4' in gpu_name:
    EST_CU_PER_HOUR = 1.94
else:
    EST_CU_PER_HOUR = 1.7

BUDGET_FILE = f'{DRIVE_DIR}/logs/cu_budget.json'
budget_state = json.load(open(BUDGET_FILE)) if os.path.isfile(BUDGET_FILE) else {'used_cu': 0.0, 'sessions': []}
cu_used = float(budget_state.get('used_cu', 0.0))
cu_remaining = max(0.0, CU_BUDGET - cu_used - STARTUP_OVERHEAD_CU)
budget_minutes = (cu_remaining / EST_CU_PER_HOUR) * 60.0 if EST_CU_PER_HOUR > 0 else 0.0
EFFECTIVE_MAX_MINUTES = min(MAX_SESSION_MINUTES, budget_minutes) if MAX_SESSION_MINUTES > 0 else budget_minutes

print(f'GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB | Compute Cap: {cap} | Native FP8: {has_native_fp8}')
print(f'Budget: {cu_used:.2f} / {CU_BUDGET:.0f} CUs used | Remaining: {cu_remaining:.2f} CUs (~{budget_minutes/60:.2f} h @ {EST_CU_PER_HOUR:.2f} CU/h)')
print('Reserved untouched: 20 CUs for YouTube extraction + 20 CUs for Pocket First-Aid Librarian')
assert cu_remaining > 1.0, f'160 CU Reddit/forum budget already reached ({cu_used:.2f} CUs used)! Stopping to preserve 20 CUs for YouTube + 20 CUs for First-Aid Librarian.'

# Auto-downgrade preset if user picked a 27B/32B model on a 24GB/16GB GPU without FP8 headroom
if vram_gb < 38 and MODEL_PRESET in ('qwen3.5-27b', 'qwen2.5-32b', 'gemma-4-31b', 'nemotron-3-nano-30b', 'qwen3-next-80b-fp8'):
    print(f'NOTE: {MODEL_PRESET} needs >= 40-80 GB VRAM; switching automatically to qwen3.5-9b for {gpu_name} ({vram_gb:.0f} GB), keeping PRIORITY_ONLY={PRIORITY_ONLY}.')
    MODEL_PRESET = 'qwen3.5-9b'

SESSION_T0 = time.time()

In [ ]:
# === CELL 2: Install vLLM and launch the model server ===
!pip -q install -U vllm

import subprocess, time, urllib.request

PRESETS = {
    'qwen3.5-9b':          {'model': 'Qwen/Qwen3.5-9B',                              'extra': ['--language-model-only', '--default-chat-template-kwargs', '{"enable_thinking": false}'], 'use_fp8': True},
    'qwen3.5-27b':         {'model': 'Qwen/Qwen3.5-27B',                             'extra': ['--language-model-only', '--default-chat-template-kwargs', '{"enable_thinking": false}'], 'use_fp8': True},
    'qwen2.5-32b':         {'model': 'Qwen/Qwen2.5-32B-Instruct',                    'extra': [], 'use_fp8': True},
    'gemma-4-31b':         {'model': 'google/gemma-4-31b-it',                        'extra': ['--language-model-only'], 'use_fp8': True},
    'gemma-4-26b-moe':     {'model': 'google/gemma-4-26b-a4b-it',                    'extra': ['--language-model-only'], 'use_fp8': True},
    'nemotron-3-nano-30b': {'model': 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16',   'extra': [], 'use_fp8': True},
    'gpt-oss-20b':         {'model': 'openai/gpt-oss-20b',                           'extra': [], 'use_fp8': False},
    'inkling-2-mini':      {'model': 'thinkingmachines/Inkling-2-Mini',              'extra': ['--trust-remote-code', '--reasoning-parser', 'inkling', '--structured-outputs-config.enable_in_reasoning=True'], 'use_fp8': True},
    'qwen3-next-80b-fp8':  {'model': 'Qwen/Qwen3-Next-80B-A3B-Instruct',             'extra': [], 'use_fp8': True},
}
cfg = PRESETS[MODEL_PRESET]
MODEL_ID = cfg['model']

# Pick concurrency based on detected Colab GPU + model size
is_small = MODEL_PRESET in ('qwen3.5-9b', 'gpt-oss-20b', 'gemma-4-26b-moe', 'inkling-2-mini')
if vram_gb >= 75:       # A100-80GB or H100
    PARALLEL = 96 if is_small else 40
elif vram_gb >= 38:     # A100-40GB
    PARALLEL = 64 if is_small else 24
elif vram_gb >= 20:     # L4 24GB
    PARALLEL = 32 if is_small else 12
else:                   # T4 16GB
    PARALLEL = 12

cmd = [
    'vllm', 'serve', MODEL_ID,
    '--host', '127.0.0.1', '--port', '8000',
    '--max-model-len', '8192',
    '--gpu-memory-utilization', '0.90',
    '--enable-prefix-caching',
    '--enable-chunked-prefill',
] + cfg['extra']

if has_native_fp8 and cfg.get('use_fp8'):
    cmd += ['--quantization', 'fp8']
elif vram_gb < 20:
    cmd += ['--dtype', 'half']

print('Launching vLLM:', ' '.join(cmd))
vllm_log = open('/content/vllm.log', 'w')
vllm_proc = subprocess.Popen(cmd, stdout=vllm_log, stderr=subprocess.STDOUT)

for i in range(240):
    if vllm_proc.poll() is not None:
        print(open('/content/vllm.log').read()[-3000:])
        raise RuntimeError('vLLM exited early — see log above')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2) as r:
            if r.status == 200:
                print(f'vLLM ready in {i*3}s! (model={MODEL_ID}, parallel={PARALLEL})')
                break
    except Exception:
        time.sleep(3)
else:
    raise RuntimeError('Timed out waiting for vLLM /health')

In [ ]:
# === CELL 3 (Recommended first): 500-thread calibration (< 1 Compute Unit) ===
# Measures exact threads/hr, JSON fail %, and quote fail % on your live Colab GPU before running all 12 makes.
import gzip, shutil, os, subprocess

os.makedirs('/content/work', exist_ok=True)
os.makedirs('/content/checkpoints', exist_ok=True)
shutil.copy2(f'{DRIVE_DIR}/colab_worker.py', '/content/colab_worker.py')

def prepare_local_db(make):
    local_db = f'/content/work/{make}.work.db'
    if os.path.isfile(local_db):
        return local_db
    drive_done = f'{DRIVE_DIR}/work_dbs/{make}.work.db'
    drive_gz   = f'{DRIVE_DIR}/inputs/{make}.work.db.gz'
    drive_raw  = f'{DRIVE_DIR}/inputs/{make}.work.db'
    if os.path.isfile(drive_done):
        print(f'[{make}] Copying existing Drive work DB -> local SSD...')
        shutil.copy2(drive_done, local_db)
    elif os.path.isfile(drive_gz):
        print(f'[{make}] Unpacking {drive_gz} -> local SSD...')
        with gzip.open(drive_gz, 'rb') as fi, open(local_db, 'wb') as fo:
            shutil.copyfileobj(fi, fo, length=1024*1024)
    elif os.path.isfile(drive_raw):
        print(f'[{make}] Copying {drive_raw} -> local SSD...')
        shutil.copy2(drive_raw, local_db)
    else:
        raise FileNotFoundError(f'Missing input for {make} in {DRIVE_DIR}/inputs/')
    return local_db

cal_make = MAKES[0]
cal_db = prepare_local_db(cal_make)
cal_args = [
    'python3', '/content/colab_worker.py',
    '--make', cal_make,
    '--work-db', cal_db,
    '--model', MODEL_ID,
    '--parallel', str(PARALLEL),
    '--limit', '500',
    '--stats-every', '250',
    '--checkpoint-jsonl', f'/content/checkpoints/{cal_make}.extract.jsonl',
    '--drive-checkpoint-jsonl', f'{DRIVE_DIR}/checkpoints/{cal_make}.extract.jsonl',
]
if PRIORITY_ONLY:
    cal_args.append('--priority-only')
subprocess.run(cal_args, check=True)
print('\nCheck Colab Resources panel for live CU/hr, optionally set CU_PER_HOUR_OVERRIDE in Cell 1, then run Cell 4!')

In [ ]:
# === CELL 4: Run Strategy A (capped strictly at 180 CUs total) and save each finished DB to Google Drive ===
import json, time, subprocess

def record_budget(elapsed_sec, note=''):
    est_cu = (elapsed_sec / 3600.0) * EST_CU_PER_HOUR
    state = json.load(open(BUDGET_FILE)) if os.path.isfile(BUDGET_FILE) else {'used_cu': 0.0, 'sessions': []}
    state['used_cu'] = round(float(state.get('used_cu', 0.0)) + est_cu, 3)
    state['sessions'].append({
        'at': time.strftime('%F %T'), 'gpu': gpu_name, 'model': MODEL_ID,
        'minutes': round(elapsed_sec / 60.0, 2), 'cu_per_hour': EST_CU_PER_HOUR,
        'est_cu': round(est_cu, 3), 'note': note,
    })
    with open(BUDGET_FILE + '.tmp', 'w') as f:
        json.dump(state, f, indent=2)
    os.replace(BUDGET_FILE + '.tmp', BUDGET_FILE)
    return state['used_cu']

t_last_log = SESSION_T0
try:
    for make in MAKES:
        elapsed_m = (time.time() - SESSION_T0) / 60.0
        rem_m = EFFECTIVE_MAX_MINUTES - elapsed_m
        if rem_m <= 2:
            print(f'Reached 180 CU budget guardrail ({EFFECTIVE_MAX_MINUTES:.1f} min cap), stopping cleanly.')
            break

        print(f'\n==================== {make.upper()} (budget left: ~{rem_m:.1f} min) ====================', flush=True)
        local_db = prepare_local_db(make)
        args = [
            'python3', '/content/colab_worker.py',
            '--make', make,
            '--work-db', local_db,
            '--model', MODEL_ID,
            '--parallel', str(PARALLEL),
            '--stats-every', '500',
            '--max-minutes', f'{rem_m:.1f}',
            '--checkpoint-jsonl', f'/content/checkpoints/{make}.extract.jsonl',
            '--drive-checkpoint-jsonl', f'{DRIVE_DIR}/checkpoints/{make}.extract.jsonl',
            '--drive-work-db', f'{DRIVE_DIR}/work_dbs/{make}.work.db',
        ]
        if PRIORITY_ONLY:
            args.append('--priority-only')
        subprocess.run(args, check=True)
        now = time.time()
        total_cu = record_budget(now - t_last_log, note=f'{make} (priority_only={PRIORITY_ONLY})')
        t_last_log = now
        print(f'Cumulative est. CU spent: {total_cu:.2f} / {CU_BUDGET:.0f} CUs (20 CUs reserved for First-Aid Librarian)', flush=True)
finally:
    now = time.time()
    if now - t_last_log > 30:
        total_cu = record_budget(now - t_last_log, note='session_end')
        print(f'Final cumulative est. CU spent: {total_cu:.2f} / {CU_BUDGET:.0f} CUs', flush=True)

print('\nBATCH COMPLETE OR STOPPED AT 180 CU CAP — ALL FINISHED DBS SAVED TO GOOGLE DRIVE!')

In [ ]:
# === CELL 5: Auto-shutdown runtime so idle GPUs never drain Compute Units ===
if AUTO_SHUTDOWN:
    try:
        vllm_proc.terminate()
    except Exception:
        pass
    from google.colab import runtime
    print('Unassigning Colab runtime to protect remaining Compute Units (incl. 20 CUs reserved for First-Aid Librarian)...')
    runtime.unassign()